# Score Random 2000 Alignments with StructuralSimilarityModel

This notebook loads the saved random-pair alignments, computes structural/semantic similarity features
using `StructuralSimilarityModel`, and saves the scored labels for downstream embedding-model training.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

/home/shayan/miniconda3/envs/story_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for c in candidates:
        if (c / '.git').exists() and (c / 'src').exists() and (c / 'data').exists():
            return c
    raise RuntimeError('Could not locate project root from current working directory.'
    )

project_root = find_project_root()
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from structural_similarity_model import StructuralSimilarityModel

alignment_path = project_root / 'data' / 'Alignment' / 'asq_random_10000_story_pair_alignments.json'
out_json = project_root / 'data' / 'Alignment' / 'asq_random_10000_pair_structural_scores.json'
out_csv = project_root / 'data' / 'Alignment' / 'asq_random_10000_pair_structural_scores.csv'

print('project_root:', project_root)
print('alignment_path exists:', alignment_path.exists(), '|', alignment_path)
print('output json:', out_json)
print('output csv :', out_csv)


project_root: /Users/shayan/Projects/NarrativeSimilarity
alignment_path exists: True | /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_10000_story_pair_alignments.json
output json: /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_10000_pair_structural_scores.json
output csv : /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_10000_pair_structural_scores.csv


In [5]:
with open(alignment_path, 'r', encoding='utf-8') as f:
    alignments = json.load(f)

print('Loaded alignment records:', len(alignments))
if len(alignments) > 0:
    print('Sample keys:', list(alignments[0].keys()))

Loaded alignment records: 10000
Sample keys: ['pair_id', 'story_a', 'story_b', 'model_output_raw', 'alignment', 'ok', 'error', 'model', 'temperature']


In [6]:
hf_cache_model_dir = Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--sentence-transformers--all-MiniLM-L6-v2'
if hf_cache_model_dir.exists():
    snapshot_dirs = sorted((hf_cache_model_dir / 'snapshots').glob('*'))
    if len(snapshot_dirs) == 0:
        raise RuntimeError(f"No local snapshots found in: {hf_cache_model_dir / 'snapshots'}")
    embedding_model_path = str(snapshot_dirs[-1])
else:
    embedding_model_path = 'sentence-transformers/all-MiniLM-L6-v2'

print('Embedding model path:', embedding_model_path)
model = StructuralSimilarityModel(embedding_model_name=embedding_model_path)

scored_rows = []
for item in tqdm(alignments, desc='Scoring alignments'):
    story_a = item.get('story_a') or {}
    story_b = item.get('story_b') or {}

    row = {
        'alignment': item.get('alignment') or {},
        'EventsA_align': story_a.get('events') or [],
        'EventsB_align': story_b.get('events') or [],
    }

    pred = model.predict_similarity(row)

    scored_rows.append({
        'pair_id': item.get('pair_id'),
        'story_a_id': story_a.get('id'),
        'story_b_id': story_b.get('id'),
        'ok': item.get('ok'),
        'error': item.get('error'),
        'num_events_a': len(row['EventsA_align']),
        'num_events_b': len(row['EventsB_align']),
        'num_matches': len((row['alignment'] or {}).get('matches', []) or []),
        'D_alignment': pred['D_alignment'],
        'alignment_similarity': pred['alignment_similarity'],
        'D_semantic': pred['D_semantic'],
        'semantic_similarity': pred['semantic_similarity'],
        'pred_event_rating_mean_joint': pred['pred_event_rating_mean_joint'],
    })

scores_df = pd.DataFrame(scored_rows)
print('Scored rows:', len(scores_df))
scores_df.head()

Embedding model path: /Users/shayan/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shayan/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring alignments:   0%|          | 0/10000 [00:00<?, ?it/s]

Scored rows: 10000


,pair_id,story_a_id,story_b_id,ok,error,num_events_a,num_events_b,num_matches,D_alignment,alignment_similarity,D_semantic,semantic_similarity,pred_event_rating_mean_joint
0,1x53zg__4g4wtc,1x53zg,4g4wtc,True,NaN,2,3,2,0.333333,0.666667,0.422465,0.577535,2.257093
1,9rprs0__3986p3,9rprs0,3986p3,True,NaN,2,1,0,1.000000,0.000000,1.000000,0.000000,1.583000
2,5w3bgs__7au7vs,5w3bgs,7au7vs,True,NaN,1,6,0,1.000000,0.000000,1.000000,0.000000,1.583000
3,3w07bi__6iaas5,3w07bi,6iaas5,True,NaN,5,5,0,1.000000,0.000000,1.000000,0.000000,1.583000
4,2schv9__3wqhqs,2schv9,3wqhqs,True,NaN,5,5,1,0.800000,0.200000,0.499647,0.500353,2.090699


In [7]:
print('Summary statistics:')
print(scores_df[['D_alignment', 'alignment_similarity', 'D_semantic', 'semantic_similarity', 'pred_event_rating_mean_joint']].describe().to_string())

print('Failed source alignments (if any):', int((scores_df['ok'] == False).sum()))

Summary statistics:
        D_alignment  alignment_similarity    D_semantic  semantic_similarity  pred_event_rating_mean_joint
count  10000.000000          10000.000000  10000.000000         10000.000000                  10000.000000
mean       0.891800              0.108200      0.865954             0.134046                      1.730052
std        0.236175              0.236175      0.258257             0.258257                      0.283191
min        0.000000              0.000000      0.113900             0.000000                      1.583000
25%        1.000000              0.000000      1.000000             0.000000                      1.583000
50%        1.000000              0.000000      1.000000             0.000000                      1.583000
75%        1.000000              0.000000      1.000000             0.000000                      1.583000
max        1.000000              1.000000      1.000000             0.886100                      2.524366
Failed source ali

In [8]:
out_json.parent.mkdir(parents=True, exist_ok=True)

scores_df.to_json(out_json, orient='records', indent=2, force_ascii=False)
scores_df.to_csv(out_csv, index=False)

print('Saved JSON:', out_json)
print('Saved CSV :', out_csv)

Saved JSON: /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_10000_pair_structural_scores.json
Saved CSV : /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_10000_pair_structural_scores.csv


## Non Random Sampling Scores

In [2]:
def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for c in candidates:
        if (c / '.git').exists() and (c / 'src').exists() and (c / 'data').exists():
            return c
    raise RuntimeError('Could not locate project root from current working directory.'
    )

project_root = find_project_root()
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from structural_similarity_model import StructuralSimilarityModel

alignment_path = project_root / 'data' / 'Alignment' / 'asq_embedding_train_v2_sample10k_story_pair_alignments.json'
out_json = project_root / 'data' / 'Alignment' / 'asq_embedding_train_v2_sample10k_pair_structural_scores.json'
out_csv = project_root / 'data' / 'Alignment' / 'asq_embedding_train_v2_sample10k_pair_structural_scores.csv'

print('project_root:', project_root)
print('alignment_path exists:', alignment_path.exists(), '|', alignment_path)
print('output json:', out_json)
print('output csv :', out_csv)


project_root: /tank/scratch/shayan/Projects/NarrativeSimilarity
alignment_path exists: True | /tank/scratch/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_embedding_train_v2_sample10k_story_pair_alignments.json
output json: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_embedding_train_v2_sample10k_pair_structural_scores.json
output csv : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_embedding_train_v2_sample10k_pair_structural_scores.csv


In [3]:
with open(alignment_path, 'r', encoding='utf-8') as f:
    alignments = json.load(f)

print('Loaded alignment records:', len(alignments))
if len(alignments) > 0:
    print('Sample keys:', list(alignments[0].keys()))

Loaded alignment records: 10000
Sample keys: ['pair_id', 'story_a', 'story_b', 'model_output_raw', 'alignment', 'ok', 'error', 'model', 'temperature', 'pair_metadata']


In [4]:
hf_cache_model_dir = Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--sentence-transformers--all-MiniLM-L6-v2'
if hf_cache_model_dir.exists():
    snapshot_dirs = sorted((hf_cache_model_dir / 'snapshots').glob('*'))
    if len(snapshot_dirs) == 0:
        raise RuntimeError(f"No local snapshots found in: {hf_cache_model_dir / 'snapshots'}")
    embedding_model_path = str(snapshot_dirs[-1])
else:
    embedding_model_path = 'sentence-transformers/all-MiniLM-L6-v2'

print('Embedding model path:', embedding_model_path)
model = StructuralSimilarityModel(embedding_model_name=embedding_model_path)

scored_rows = []
for item in tqdm(alignments, desc='Scoring alignments'):
    story_a = item.get('story_a') or {}
    story_b = item.get('story_b') or {}

    row = {
        'alignment': item.get('alignment') or {},
        'EventsA_align': story_a.get('events') or [],
        'EventsB_align': story_b.get('events') or [],
    }

    pred = model.predict_similarity(row)

    scored_rows.append({
        'pair_id': item.get('pair_id'),
        'story_a_id': story_a.get('id'),
        'story_b_id': story_b.get('id'),
        'ok': item.get('ok'),
        'error': item.get('error'),
        'num_events_a': len(row['EventsA_align']),
        'num_events_b': len(row['EventsB_align']),
        'num_matches': len((row['alignment'] or {}).get('matches', []) or []),
        'D_alignment': pred['D_alignment'],
        'alignment_similarity': pred['alignment_similarity'],
        'D_semantic': pred['D_semantic'],
        'semantic_similarity': pred['semantic_similarity'],
        'pred_event_rating_mean_joint': pred['pred_event_rating_mean_joint'],
    })

scores_df = pd.DataFrame(scored_rows)
print('Scored rows:', len(scores_df))
scores_df.head()

Embedding model path: /home/shayan/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf


Scoring alignments: 100%|██████████| 10000/10000 [00:13<00:00, 742.90it/s]

Scored rows: 10000


,pair_id,story_a_id,story_b_id,ok,error,num_events_a,num_events_b,num_matches,D_alignment,alignment_similarity,D_semantic,semantic_similarity,pred_event_rating_mean_joint
0,4aglga__csv_text_b_f3a1afb22aa0,4aglga,csv_text_b_f3a1afb22aa0,True,None,1,5,0,1.00000,0.00000,1.000000,0.000000,1.583000
1,4hzsyu__csv_text_b_bbf9dec777ac,4hzsyu,csv_text_b_bbf9dec777ac,True,None,3,5,0,1.00000,0.00000,1.000000,0.000000,1.583000
2,csv_text_a_928349d5e2ae__csv_text_b_58da0e243459,csv_text_a_928349d5e2ae,csv_text_b_58da0e243459,True,None,10,4,0,1.00000,0.00000,1.000000,0.000000,1.583000
3,csv_text_a_b97adb970f1a__csv_text_b_7a7d21cb45bd,csv_text_a_b97adb970f1a,csv_text_b_7a7d21cb45bd,True,None,11,8,0,1.00000,0.00000,1.000000,0.000000,1.583000
4,csv_text_a_bab34bf423f7__6pjdlk,csv_text_a_bab34bf423f7,6pjdlk,True,None,8,3,3,0.03125,0.96875,0.403362,0.596638,2.335984


In [5]:
print('Summary statistics:')
print(scores_df[['D_alignment', 'alignment_similarity', 'D_semantic', 'semantic_similarity', 'pred_event_rating_mean_joint']].describe().to_string())

print('Failed source alignments (if any):', int((scores_df['ok'] == False).sum()))

Summary statistics:
        D_alignment  alignment_similarity    D_semantic  semantic_similarity  pred_event_rating_mean_joint
count  10000.000000          10000.000000  10000.000000         10000.000000                  10000.000000
mean       0.925253              0.074747      0.930750             0.069250                      1.662779
std        0.219998              0.219998      0.196150             0.196150                      0.225750
min        0.000000              0.000000      0.049696             0.000000                      1.583000
25%        1.000000              0.000000      1.000000             0.000000                      1.583000
50%        1.000000              0.000000      1.000000             0.000000                      1.583000
75%        1.000000              0.000000      1.000000             0.000000                      1.583000
max        1.000000              1.000000      1.000000             0.950304                      2.503362
Failed source ali

In [6]:
out_json.parent.mkdir(parents=True, exist_ok=True)

scores_df.to_json(out_json, orient='records', indent=2, force_ascii=False)
scores_df.to_csv(out_csv, index=False)

print('Saved JSON:', out_json)
print('Saved CSV :', out_csv)

Saved JSON: /tank/scratch/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_embedding_train_v2_sample10k_pair_structural_scores.json
Saved CSV : /tank/scratch/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_embedding_train_v2_sample10k_pair_structural_scores.csv
